# Small Triangular HMC Stage Notebook

This notebook reads a rendered stage report directory produced by `test/archive_small_benchmark/render_small_hmc_stage_report.py`.
It is meant for interactive inspection of the staged small-parameter correctness campaign.


In [ ]:
from pathlib import Path
import base64
import re
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

REPORT_DIR = Path("../../data/triangular_hmc_small_benchmark/full_grid_progress_v3")

def render_report_markdown(report_dir: Path) -> Markdown:
    text = (report_dir / "report.md").read_text(encoding="utf-8")

    def repl(match: re.Match[str]) -> str:
        alt_text, rel_path = match.groups()
        image_path = (report_dir / rel_path).resolve()
        if image_path.suffix.lower() == ".png" and image_path.exists():
            encoded = base64.b64encode(image_path.read_bytes()).decode("ascii")
            return f'<div><img alt="{alt_text}" src="data:image/png;base64,{encoded}" style="max-width: 100%;"/></div>'
        return match.group(0)

    text = re.sub(r"!\[([^\]]*)\]\(([^)]+)\)", repl, text)
    return Markdown(text)

cases = pd.read_csv(REPORT_DIR / "stage_cases.csv")
observables = pd.read_csv(REPORT_DIR / "stage_observables.csv")
samples = pd.read_csv(REPORT_DIR / "stage_samples.csv")
if "case_id" not in cases.columns:
    cases["case_id"] = cases["name"]
if "case_id" not in observables.columns:
    observables["case_id"] = observables["name"]
if "case_id" not in samples.columns:
    samples["case_id"] = samples["name"]

cases = cases.sort_values(["Nbos", "U2"]).reset_index(drop=True)
observables = observables.sort_values(["Nbos", "U2", "observable"]).reset_index(drop=True)
samples = samples.sort_values(["Nbos", "U2", "mode", "observable", "sample_index"]).reset_index(drop=True)
REPORT_DIR


## Definitions

- `tau_int(doubleOcc)`: integrated autocorrelation time estimated from the post-cut `doubleOcc` series.
  Larger values mean slower mixing.
- `ESS / sec`: effective sample size per second, estimated from the same `doubleOcc` series.
  Larger values mean more statistically independent samples per wall-clock second.
- `Speed ratio`: `ESS/sec(HMC) / ESS/sec(Local)`.
- In the rendered report, `Nbos` is encoded by color and `local/HMC` by line style.


In [ ]:
display(render_report_markdown(REPORT_DIR))


In [ ]:
cases[[
    "name",
    "Nbos",
    "U2",
    "acceptance_mean",
    "local_tau_int_doubleOcc",
    "hmc_tau_int_doubleOcc",
    "local_ess_per_sec_doubleOcc",
    "hmc_ess_per_sec_doubleOcc",
    "speed_ratio_hmc_over_local",
    "passed",
]].sort_values(["Nbos", "U2"])


In [ ]:
display(Image(filename=str(REPORT_DIR / "overview.png")))


## Observable Tables

Set `OBS_NAME` and optionally `NBOS_SELECT` to inspect the benchmark table directly.


In [ ]:
OBS_NAME = "doubleOcc"
NBOS_SELECT = None  # e.g. 10, 100, 1000

obs_table = observables[observables["observable"] == OBS_NAME].copy()
if NBOS_SELECT is not None:
    obs_table = obs_table[obs_table["Nbos"] == NBOS_SELECT]
obs_table[[
    "name",
    "Nbos",
    "U2",
    "local_mean",
    "local_err",
    "hmc_mean",
    "hmc_err",
    "z_score",
    "passed",
]].sort_values(["Nbos", "U2"])


In [ ]:
nbos_values = [NBOS_SELECT] if NBOS_SELECT is not None else sorted(cases["Nbos"].unique())
for obs_name in ["doubleOcc", "squareOcc", "nearestOcc", "IPR", "PF_Gamma", "kinetic"]:
    for nbos in nbos_values:
        path = REPORT_DIR / f"observable_{obs_name}_n{int(nbos)}.png"
        if path.exists():
            display(Markdown(f"## {obs_name}, Nbos={int(nbos)}"))
            display(Image(filename=str(path)))


## Sample Traces

This cell replots the raw `repeat=0` benchmark traces from `stage_samples.csv`, so you can change the case and observable without rerendering the report.


In [ ]:
CASE_NAME = cases.loc[0, "name"]
CASE_ID = cases.loc[0, "case_id"]
TRACE_REPEAT = 0
TRACE_OBSERVABLES = ["doubleOcc", "nearestOcc", "IPR", "SF_Gamma"]
trace_df = samples[(samples["case_id"] == CASE_ID) & (samples["repeat"] == TRACE_REPEAT)]
thermal_cut = int(trace_df["thermal_cut"].iloc[0])

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
axes = axes.flatten()
for ax, obs_name in zip(axes, TRACE_OBSERVABLES):
    obs_df = trace_df[trace_df["observable"] == obs_name]
    for mode_name, style in [("local", "-"), ("hmc", "--")]:
        mode_df = obs_df[obs_df["mode"] == mode_name]
        ax.plot(mode_df["sample_index"], mode_df["value"], style, lw=1.0, label=mode_name)
    ax.axvline(thermal_cut, color="black", linestyle=":", lw=1.0)
    ax.set_title(obs_name)
axes[0].legend()
fig.suptitle(f"{CASE_ID}, repeat={TRACE_REPEAT}")
fig.tight_layout()
plt.show()


## Rendered Case Assets

This cell shows the rendered PNGs for a chosen case, including the tuning evidence figure.


In [ ]:
CASE_NAME = cases.loc[0, "name"]
CASE_ID = cases.loc[0, "case_id"]
trace_paths = sorted(REPORT_DIR.glob(f"trace_{CASE_ID}_rep*.png"))
if not trace_paths:
    trace_paths = sorted(REPORT_DIR.glob(f"trace_{CASE_NAME}_rep*.png"))
tune_path = REPORT_DIR / f"tune_{CASE_ID}.png"
if not tune_path.exists():
    tune_path = REPORT_DIR / f"tune_{CASE_NAME}.png"
display(Markdown(f"## {CASE_ID}"))
for trace_path in trace_paths:
    display(Image(filename=str(trace_path)))
if tune_path.exists():
    display(Image(filename=str(tune_path)))
